# 🍅 Teaching a Computer to Read Movie Reviews

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/JulesMalin/isba2411-nlp/blob/main/Week%203/rotten_tomatoes_sentiment/rotten_tomatoes_sentiment.ipynb)

### ISBA 2411 · Week 3 · Supervised Text Classification

In the next few minutes we'll build a model that reads a movie review and decides whether it's
**positive** 👍 or **negative** 👎 — using two of the most important classic tools in NLP:

> **TF-IDF** (turn words into numbers) &nbsp;+&nbsp; **Logistic Regression** (learn what those numbers mean)

No GPU. No transformers. It trains in **seconds** and gets **~79%** of reviews right. Then we'll
crack it open to see *what it learned* — and find the one thing that fools it. Let's go!

## 1. Setup
Install the four libraries we need. On Colab this takes a few seconds.

In [ ]:
!pip install -q datasets scikit-learn joblib matplotlib

## 2. The data: 10,000 real movie reviews

The [`rotten_tomatoes`](https://huggingface.co/datasets/rotten_tomatoes) dataset is a classic
benchmark: short critic blurbs, each labeled **0 = negative** or **1 = positive**. It's already
split into train / validation / test sets for us.

In [ ]:
from datasets import load_dataset

ds = load_dataset("rotten_tomatoes")
print(ds)

print(f"\nTraining reviews:  {len(ds['train']):,}")
print(f"Test reviews:      {len(ds['test']):,}")

Let's actually **read a few** so we know what the model is up against:

In [ ]:
LABELS = ["negative 👎", "positive 👍"]

for i in [0, 1, 5000, 8000]:
    row = ds["train"][i]
    print(f"[{LABELS[row['label']]}]  {row['text']}\n")

Is the dataset balanced? (If one class dominates, accuracy can be misleading.)

In [ ]:
from collections import Counter

counts = Counter(ds["train"]["label"])
print(f"Negative reviews: {counts[0]:,}")
print(f"Positive reviews: {counts[1]:,}")
print("\n→ A perfect 50/50 split, so plain accuracy is a fair scorecard.")

## 3. Build the model

Two steps, chained into a single scikit-learn **`Pipeline`**:

1. **`TfidfVectorizer`** — turns each review into a vector of word (and 2-word phrase) weights.
   Distinctive words like *"masterpiece"* get a high weight; ubiquitous words get a low one.
2. **`LogisticRegression`** — learns a weight for each feature: positive weights vote 👍,
   negative weights vote 👎.

🧠 **A choice worth pausing on:** we *keep* stop words like **"not," "no," "never."** Most text
pipelines throw these away — but in *sentiment* they flip a sentence's meaning (*"not good"*!),
so dropping them actually **costs us ~3 points** of accuracy. Little decisions matter.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

model = Pipeline([
    ("tfidf", TfidfVectorizer(
        lowercase=True,
        stop_words=None,        # keep "not"/"no"/"never" — they carry sentiment
        ngram_range=(1, 2),     # single words + two-word phrases
        min_df=2,               # ignore words that appear in only one review
        sublinear_tf=True,
    )),
    ("clf", LogisticRegression(max_iter=1000, C=3.0)),
])
model

### Train it
This is the whole show — one call to `.fit()`:

In [ ]:
model.fit(ds["train"]["text"], ds["train"]["label"])

n_features = len(model.named_steps["tfidf"].vocabulary_)
print(f"✅ Trained! The model learned weights for {n_features:,} words & phrases.")

## 4. How good is it?

The real test: reviews the model has **never seen**.

In [ ]:
from sklearn.metrics import accuracy_score, classification_report

preds = model.predict(ds["test"]["text"])
acc = accuracy_score(ds["test"]["label"], preds)

print(f"🎯 Test accuracy: {acc:.1%}\n")
print(classification_report(ds["test"]["label"], preds,
                            target_names=["negative", "positive"]))

A **confusion matrix** shows *where* the mistakes happen — false positives vs. false negatives:

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay

fig, ax = plt.subplots(figsize=(4.5, 4.5))
ConfusionMatrixDisplay.from_predictions(
    ds["test"]["label"], preds,
    display_labels=["negative", "positive"],
    cmap="Reds", colorbar=False, ax=ax,
)
ax.set_title(f"Confusion Matrix  —  {acc:.1%} accuracy")
plt.tight_layout()
plt.show()

## 5. What did the model actually *learn*? 🔍

This is the fun part. Because it's a linear model, every word has a single weight we can read
straight off. The most **positive** and most **negative** words are literally what the model
thinks "good review" and "bad review" sound like.

In [ ]:
import numpy as np

feature_names = np.array(model.named_steps["tfidf"].get_feature_names_out())
weights = model.named_steps["clf"].coef_[0]

order = weights.argsort()
top_neg = order[:12]           # most negative
top_pos = order[-12:]          # most positive

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 5))

ax1.barh(feature_names[top_neg], weights[top_neg], color="#d64545")
ax1.set_title("Words that scream 👎 NEGATIVE")
ax1.invert_yaxis()

ax2.barh(feature_names[top_pos], weights[top_pos], color="#2a9d5c")
ax2.set_title("Words that scream 👍 POSITIVE")
ax2.invert_yaxis()

plt.tight_layout()
plt.show()

## 6. Try it yourself! 🎬

Here's a tiny helper that runs any sentence through the model and shows how **confident** it is.

In [ ]:
def predict(sentence):
    idx = int(model.predict([sentence])[0])
    conf = model.predict_proba([sentence])[0][idx]
    emoji = "👍" if idx == 1 else "👎"
    label = "POSITIVE" if idx == 1 else "NEGATIVE"
    print(f"{emoji}  {label:8s} ({conf:.0%} confident)   {sentence!r}")

for s in [
    "A dazzling, heartfelt triumph that earns every tear.",
    "Boring, predictable, and painfully long.",
    "The performances are electric and the script crackles with wit.",
    "A dull, lifeless mess from start to finish.",
]:
    predict(s)

Now **your turn** — run this cell and type your own review (great for audience suggestions!):

In [ ]:
review = input("Type a movie review: ")
predict(review)

## 7. The catch: it doesn't really understand word *order* ⚠️

TF-IDF is a **"bag of words"** — it mostly sees *which* words appear, not the order. So negation
can trip it up. Watch:

In [ ]:
for s in [
    "good",
    "not good",
    "this movie is not good at all",
]:
    predict(s)

print("\n💡 See how 'not' doesn't fully rescue it? Word order is exactly what next")
print("   week's transformer models are built to capture.")

## 8. Save the model 💾

Save the whole pipeline to a single file so `predict.py` (or a future app) can reload it
instantly — no retraining.

In [ ]:
import joblib

joblib.dump(model, "model.joblib")
print("Saved → model.joblib")

# On Colab, uncomment to download it to your computer:
# from google.colab import files; files.download("model.joblib")

## 🎉 Recap

In a few minutes and ~30 lines of code we:

- turned raw text into numbers with **TF-IDF**,
- trained a **Logistic Regression** classifier to **~79%** accuracy,
- **looked inside** to see the words it learned, and
- found its **blind spot** (word order) — the perfect on-ramp to transformers.

**A strong, honest baseline is always step one.** Next week we'll see how much better we can do. 🚀